# Titanic 데이터 분류 모델 (DecisionTreeClassifier, Lv2)

seaborn `titanic` 데이터로 **생존 여부(survived)** 를 예측하는 분류 예제입니다.

- **X (독립변수)**: `pclass`, `fare`, `age`, `embarked`
- **y (종속변수)**: `survived` (0=사망, 1=생존)
- **전처리**:
  - 수치형: `SimpleImputer` + `StandardScaler`
  - 범주형: `SimpleImputer` + `OneHotEncoder`
- **모델**: `ColumnTransformer` + `DecisionTreeClassifier`


# 환경설정

In [ ]:
# 필요한 라이브러리 import

import seaborn as sns
import pandas as pd
import numpy as np

# train_test_split: 데이터를 학습용 / 테스트용으로 나누기 위해 사용
from sklearn.model_selection import train_test_split


from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

# 분류 성능 평가 지표
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# 데이터 불러오기

In [ ]:
# ============================================================
# 1. 데이터 로드
# ============================================================
titanic = sns.load_dataset('titanic')

print('데이터 shape:', titanic.shape)
print()
titanic.loc[:, ['survived', 'pclass', 'fare', 'age', 'embarked']].head()


데이터 shape: (891, 15)



,survived,pclass,fare,age,embarked
0,0,3,7.2500,22.0,S
1,1,1,71.2833,38.0,C
2,1,3,7.9250,26.0,S
3,1,1,53.1000,35.0,S
4,0,3,8.0500,35.0,S


# 데이터 처리

In [ ]:
# ============================================================
# 2. 전처리 / 학습·테스트 분리
# ============================================================
# 사용할 독립변수(X)
# pclass   : 객실 등급 (1등석, 2등석, 3등석)
# fare     : 승객이 지불한 요금
# age      : 승객의 나이 (결측치가 일부 존재)
# embarked : 탑승 항구 (C=Cherbourg, Q=Queenstown, S=Southampton)
# 코드

# 종속변수(y)
# survived : 생존 여부 (0=사망, 1=생존)
# 코드

# 80% 학습, 20% 테스트 (random_state=42 로 재현 가능)

print(f'학습 데이터: {len(X_train)}건')
print(f'테스트 데이터: {len(X_test)}건')
print()
print('결측치 개수 확인:')
print(X.isnull().sum())


학습 데이터: 712건
테스트 데이터: 179건

결측치 개수 확인:
pclass        0
fare          0
age         177
embarked      2
dtype: int64


# 데이터 학습 전 : Pipeline 정의

In [ ]:
# ============================================================
# 3. ColumnTransformer + Pipeline 정의
# ============================================================
# 수치형 변수와 범주형 변수를 나눔
# pclass, fare, age 는 숫자형 데이터
# embarked 는 문자형(범주형) 데이터
# 코드

# 수치형 전처리
# 1) SimpleImputer(strategy='median')
#    -> age 같은 숫자형 결측치를 중앙값으로 채움
# 2) StandardScaler()
#    -> 변수 크기를 평균 0, 표준편차 1 기준으로 맞춤
# 코드

# 범주형 전처리
# 1) SimpleImputer(strategy='most_frequent')
#    -> embarked 결측치를 가장 많이 나온 값으로 채움
# 2) OneHotEncoder(handle_unknown='ignore')
#    -> C / Q / S 같은 문자값을 0과 1로 분리된 컬럼으로 변환
#    -> 예: embarked_C, embarked_Q, embarked_S
# 코드


# ColumnTransformer
# -> 어떤 컬럼에는 수치형 전처리, 어떤 컬럼에는 범주형 전처리를 적용할지 지정
# 코드

# 최종 Pipeline
# 1) preprocessor에서 전처리 수행
# 2) 전처리된 데이터를 DecisionTreeClassifier에 전달하여 분류 학습
# 코드

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['pclass', 'fare', 'age']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['embarked'])])),
                ('model', DecisionTreeClassifier(random_state=42))])

# 데이터 학습

In [ ]:
# 4. 학습

# 코드

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['pclass', 'fare', 'age']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['embarked'])])),
                ('model', DecisionTreeClassifier(random_state=42))])

# 예측값 산출 및 평가지표 설정

In [ ]:
# 5. 평가 (테스트 데이터 기준)

y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('--- 테스트 데이터 평가 결과 ---')
print(f'Accuracy  (정확도)        : {accuracy:.4f}')
print(f'Precision (정밀도)        : {precision:.4f}')
print(f'Recall    (재현율)        : {recall:.4f}')
print(f'F1 score  (F1 점수)       : {f1:.4f}')


--- 테스트 데이터 평가 결과 ---
Accuracy  (정확도)        : 0.6760
Precision (정밀도)        : 0.6429
Recall    (재현율)        : 0.4865
F1 score  (F1 점수)       : 0.5538


# 사용자 입력 테스트

In [ ]:
# 6. 사용자 입력 예측



객실 등급(pclass, 1~3)을 입력하세요: 3
요금(fare)을 입력하세요: 30
나이(age)를 입력하세요: 20
탑승 항구(embarked: C / Q / S)를 입력하세요: C

예측 결과(survived): 0 (사망)
